In [1]:
import numpy as np
from numba import jit, njit, typed, types
import timeit

In [2]:


# ---------- helper Numba types ----------
coord_type    = types.UniTuple(types.int16, 2)        # (row, col)
list_type     = types.ListType(coord_type)            # list of coordinates
array2d_type  = types.int16[:, ::1]                   # (2, n_pts) C-contiguous
dict_lists_t  = types.DictType(types.int64, list_type)
dict_arrays_t = types.DictType(types.int64, array2d_type)

@jit
def extract_cloud_coordinates(cloudtracknumber_field,   # 3-D, shape (1, ny, nx)
                              cloud_id_in_field,        # 1-D array of unique IDs
                              max_size):                # per-cloud hard cap
    """
    Returns a Dict[int -> int16[:, ::1]]
        key   : cloud ID
        value : 2×N array with the exact #pixels (N ≤ max_size)
                 row coords in axis=0, col coords in axis=1
    Memory use ≈ Σ( N_cloud × 2 × 2 bytes ) with zero over-allocation.
    """

    # -- first pass: collect coordinates in typed.Lists --------------------
    coord_lists = typed.Dict.empty(                     # type: Dict[int, List[(int16,int16)]]
        key_type   = types.int64,
        value_type = list_type
    )

    ny, nx = cloudtracknumber_field.shape[1:]

    for row in range(ny):
        for col in range(nx):
            cid = cloudtracknumber_field[0, row, col]
            if cid == 0:
                continue          # background pixel – ignore

            if cid not in coord_lists:
                coord_lists[cid] = typed.List.empty_list(coord_type)

            lst = coord_lists[cid]
            if len(lst) < max_size:              # honour the user-supplied cap
                lst.append((np.int16(row), np.int16(col)))

    # -- second pass: pack each list into a perfectly-sized 2×N array ------
    result = typed.Dict.empty(                     # type: Dict[int, int16[:,::1]]
        key_type   = types.int64,
        value_type = array2d_type
    )

    for cid in coord_lists:
        lst = coord_lists[cid]
        n   = len(lst)
        arr = np.empty((2, n), dtype=np.int16)

        for i in range(n):
            rc = lst[i]
            arr[0, i] = rc[0]     # row
            arr[1, i] = rc[1]     # col

        result[cid] = arr

    return result


/tmp/ipykernel_53115/2565604207.py:8: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @jit


In [5]:
test_arr = np.zeros((1, 5, 5), dtype=np.int16)
test_arr[0, 1:2, 1:3] = 1
test_arr[0, 3:5, 2:5] = 2
test_arr

array([[[0, 0, 0, 0, 0],
        [0, 1, 1, 0, 0],
        [0, 0, 0, 0, 0],
        [0, 0, 2, 2, 2],
        [0, 0, 2, 2, 2]]], dtype=int16)

In [6]:
result = extract_cloud_coordinates(test_arr,np.array([1,2]), 10)

KeyboardInterrupt: 